# M2 · Inside the machine 1: characters become vectors

**Tiny Transformer, episode M2.** So far our tiny transformer was a closed box: 32 ids in, 65 probabilities out. Here you open
it and run its first stage yourself, on the trained model and our running line, *“You taught me language; and my p”*. An id is only a label (37
is not “more” than 36), so the model swaps each id for a list of 28 learned numbers it can add and multiply: the
token embedding table says what each character is, the position embedding says where it sits, and the 32 rows of 28
numbers that leave are all the rest of the machine ever sees.

[Open in Google Colab](https://colab.research.google.com/github/gkiri/tiny_repo/blob/main/tiny-transformer/notebooks/M2-inside-embeddings.ipynb) (execution uses Google's hosted runtime; no graphics card needed). On your own computer:
`pip install -r requirements.txt`, then `jupyter notebook M2-inside-embeddings.ipynb`.

Every number below is computed here, from the trained model's 9,417 learned numbers (`prompter.json`, the recorded training
run at its best step). Where the course's page shows a recorded number from the same run, a cell asserts that this notebook
computed the same one, to 4 decimals.

Laid out like the page: two sections, five chapters, and under each chapter its page's three steps (Picture it, Watch it on
our line, Read and run the code), then the code to run and a "your turn".

In [1]:
#@title Setup: run me first (one click) { display-mode: "form" }
# In Google Colab this fetches the course's labs (notebooks, data and the trained model) and moves into this notebook's
# folder, so every file is where the cells below look for it. On your own computer, inside a copy of the labs, it does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/tiny_repo"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/gkiri/tiny_repo", "/content/tiny_repo"], check=True)
    os.chdir("/content/tiny_repo/tiny-transformer/notebooks")
print("ready:", os.path.basename(os.getcwd()))

ready: notebooks


In [2]:
import base64, json, math, os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
print(torch.__version__)

2.10.0+cpu


**The model's code.** The next cell is the model's whole file, `training/prompter/model.py`, copied in unchanged. The one
after it is this episode's teaching code, `code/inside_embeddings.py`: five small functions that do this stage one step at a
time, on the model's own parts.

In [3]:
"""Our tiny transformer: 9,417 learned numbers that predict the next character of Shakespeare.

It reads up to 32 characters of text and gives a probability to each of the 65 characters that could come next. Writing is
that prediction in a loop: pick a character, append it, slide the 32-character window, and predict again. (The class is
called Prompter for the course's file names; learners meet it as "our tiny transformer".)

The architecture is the "10k" configuration of maxpolaczuk/tiny-character-transformer (a 1-block, 2-head character GPT on Tiny
Shakespeare). The code here is written fresh for the course (that repository has no licence, so none of its code or weights
are copied): every line is one the course teaches.

    characters ─► token table (65 × 28) + position table (32 × 28)        lookup:      what, and where
               ─► one block: LayerNorm ─► 2-head causal attention ─► +    look back:   which earlier letters matter
                             LayerNorm ─► feed-forward 28 → 56 → 28 ─► +  think:       transform each row on its own
               ─► final LayerNorm ─► readout with the token table again   score all 65 characters
               ─► softmax                                                  probabilities for the next character

Parameters: token table 1,820 · positions 896 · block 6,580 (two LayerNorms 112, q|k|v 2,436, attention out 812,
feed-forward 1,624 + 1,596) · final LayerNorm 56 · readout bias 65 = 9,417.
"""

import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass(frozen=True)
class Config:
    vocab_size: int = 65    # every distinct character in Tiny Shakespeare
    block_size: int = 32    # how many characters the model can read at once (its context)
    d_model: int = 28       # how many numbers describe one character in one place (a row)
    n_head: int = 2         # attention heads, each looking with 28 / 2 = 14 numbers
    d_ff: int = 56          # the feed-forward network's hidden width (2 × d_model)
    n_layer: int = 1        # transformer blocks

    def to_dict(self):
        return asdict(self)


class Tokenizer:
    """One id per character, in sorted order: '\\n' is 0, ' ' is 1, … 'z' is 64."""

    def __init__(self, text: str):
        self.chars = sorted(set(text))
        self.stoi = {c: i for i, c in enumerate(self.chars)}

    def encode(self, s: str) -> list[int]:
        return [self.stoi[c] for c in s]

    def decode(self, ids) -> str:
        return "".join(self.chars[int(i)] for i in ids)


class CausalSelfAttention(nn.Module):
    """Each row asks a question (q), every earlier row offers a label (k) and a message (v). A row reads the messages of the
    rows whose labels match its question best, never a row to its right."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.n_head, self.head_dim = cfg.n_head, cfg.d_model // cfg.n_head
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model)     # q, k and v for both heads in one matrix
        self.proj = nn.Linear(cfg.d_model, cfg.d_model)        # mix the heads' results back into one row
        mask = torch.tril(torch.ones(cfg.block_size, cfg.block_size))
        self.register_buffer("mask", mask.view(1, 1, cfg.block_size, cfg.block_size), persistent=False)

    def forward(self, x, keep=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, self.head_dim).transpose(1, 2) for t in (q, k, v))
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)          # how well each question matches each label
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))   # no peeking at the future
        att = F.softmax(att, dim=-1)                                          # shares that add up to 1
        if keep is not None:
            keep["attn"] = att.detach()
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)            # each row: a weighted mix of messages
        return self.proj(y)


class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ff = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(), nn.Linear(cfg.d_ff, cfg.d_model))

    def forward(self, x, keep=None):
        x = x + self.attn(self.ln1(x), keep)    # look back, and add what was found to the row
        x = x + self.ff(self.ln2(x))            # think about each row on its own, and add that too
        return x


class Prompter(nn.Module):
    def __init__(self, cfg: Config = Config()):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)    # one learned row per character
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)    # one learned row per place 0 … 31
        self.blocks = nn.Sequential(*[Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head_bias = nn.Parameter(torch.zeros(cfg.vocab_size))  # the readout reuses tok_emb (tied); only its bias is new
        self.apply(self._init)

    @staticmethod
    def _init(m):
        # small random numbers (σ = 0.02), so the untrained model's guesses are nearly even: its first loss is about
        # ln 65 = 4.17, the loss of a uniform guess over 65 characters
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, keep=None):
        T = idx.size(1)
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))   # what + where
        for b in self.blocks:
            x = b(x, keep)
        x = self.ln_f(x)
        logits = x @ self.tok_emb.weight.T + self.head_bias                         # a score for every character
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, n, temperature=1.0, generator=None):
        for _ in range(n):
            logits, _ = self(idx[:, -self.cfg.block_size:])
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            nxt = torch.multinomial(probs, 1, generator=generator)
            idx = torch.cat([idx, nxt], dim=1)
        return idx


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

In [4]:
"""M2 · Inside the machine 1: characters become vectors (ADR 0011). The first stage of our tiny transformer, done one step at a
time on top of training/prompter/model.py's own parts:

  learned_numbers(model)      how many learned numbers sit in each part of the model (they add up to 9,417)
  token_rows(model, ids)      each id fetches its entry of the token embedding table: [T] → [T×28]
  position_rows(model, T)     the position embedding's entries for positions 0 … T−1: [T×28]
  embed(model, ids)           token rows + position rows: the rows that enter the block, [T×28]
  nearest(model, tok, ch, k)  the k entries of the token table closest in direction to ch's entry (cosine similarity)

code/test_inside_embeddings.py checks each one against model.py's own forward pass and the recorded run (prompter-stages.json,
prompter-embeddings.json); notebooks/make_m2_inside_embeddings.py copies this file into M2-inside-embeddings.ipynb."""
import torch


def learned_numbers(model):
    """The model's learned numbers, part by part, in the order the forward pass meets them."""
    count = lambda module: sum(p.numel() for p in module.parameters())   # every weight and bias in one part
    return {
        "token table": model.tok_emb.weight.numel(),        # [65 × 28]
        "position table": model.pos_emb.weight.numel(),     # [32 × 28]
        "block": count(model.blocks),                       # two LayerNorms, attention, feed-forward
        "final LayerNorm": count(model.ln_f),               # a scale and a shift for each of the 28
        "readout bias": model.head_bias.numel(),            # [65]: the readout reuses the token table, plus these
    }


def token_rows(model, ids):
    """Each id fetches its entry of the token embedding table."""
    ids = torch.as_tensor(ids)                  # [T]: one id per character
    table = model.tok_emb.weight                # [65 × 28]: one learned entry per character
    return table[ids]                           # [T × 28]: row t is entry ids[t]


def position_rows(model, T):
    """The position embedding's first T entries: one per position, 0 … T−1."""
    table = model.pos_emb.weight                # [32 × 28]: one learned entry per position
    return table[:T]                            # [T × 28]: row t is entry t


def embed(model, ids):
    """The rows that leave the first stage: what each character is, plus where it sits."""
    rows = token_rows(model, ids)               # [T × 28]
    return rows + position_rows(model, len(rows))   # [T × 28]: added number by number


def nearest(model, tok, ch, k=3):
    """The k entries of the token table that point most nearly the same way as ch's entry, with their cosine similarity
    (1 = the same direction, 0 = unrelated)."""
    table = model.tok_emb.weight.detach()                    # [65 × 28]
    unit = table / table.norm(dim=1, keepdim=True)           # [65 × 28]: every entry scaled to length 1
    cos = unit @ unit[tok.stoi[ch]]                          # [65]: each entry's cosine with ch's entry
    cos[tok.stoi[ch]] = -2                                   # leave out ch itself
    top = cos.topk(k)                                        # the k largest
    return [(tok.chars[i], round(c, 4)) for c, i in zip(top.values.tolist(), top.indices.tolist())]

**The trained numbers.** In Colab the Setup cell fetched them with the labs; inside a copy of the course they are found there.

In [5]:
here = os.path.abspath('')
paths = [os.path.join(here, '..', 'model', 'prompter.json'),                                                     # in the labs
         os.path.join(here, '..', '..', '..', '..', 'src', 'specimen', 'tiny-transformer', 'data', 'prompter.json')]   # in the course
found = [p for p in paths if os.path.exists(p)]
assert found, "the model's trained numbers are not here: run the Setup cell at the top"
exp = json.load(open(found[0]))
state = {t['name']: torch.tensor(np.frombuffer(base64.b64decode(t['data']), dtype='<f4').reshape(t['shape']).copy()) for t in exp['weights']['tensors']}
cfg = Config(**exp['config'])
model = Prompter(cfg)
model.load_state_dict(state)
model.eval()
torch.set_grad_enabled(False)                  # this notebook only reads the model; nothing here trains it
tok = Tokenizer(exp['vocab'])                  # the 65 characters, in id order
line = 'You taught me language; and my p'
ids = torch.tensor(tok.encode(line))
print('trained numbers from step', exp['checkpoint_step'], ':', count_params(model), 'learned numbers')
print(repr(line), '→ ids', tuple(ids.shape))
print(ids.tolist())
assert ''.join(tok.chars) == exp['vocab'] and exp['checkpoint_step'] == 98000 and tuple(ids.shape) == (32,)
assert ids.tolist() == [37, 53, 59, 1, 58, 39, 59, 45, 46, 58, 1, 51, 43, 1, 50, 39, 52, 45, 59, 39, 45, 43, 11, 1, 39, 52, 42, 1, 51, 63, 1, 54]

trained numbers from step 98000 : 9417 learned numbers
'You taught me language; and my p' → ids (32,)
[37, 53, 59, 1, 58, 39, 59, 45, 46, 58, 1, 51, 43, 1, 50, 39, 52, 45, 59, 39, 45, 43, 11, 1, 39, 52, 42, 1, 51, 63, 1, 54]


# Section 1 · Into the box

The box opened at its first stage: the plan of every stage, the token table that turns ids into rows, and what training put in
it. Each chapter below follows its page: Picture it, Watch it on our line, then Read and run the code.

## M2.1 · The plan inside

*What is inside the box?*

**Predict first** (the page asks it too, before its clip): our tiny transformer has 9,417 learned numbers. Where do most of
them sit: spread about evenly, a third each in the tables, the block and the readout; in the one block; or in the readout, which
scores all 65 characters?

### M2.1.1 · Picture it

A plan shows what each stage hands on, and what it holds. In a made-up machine, 3 letters go in: shape `[3]`. A table turns
each into 2 numbers, `[3×2]`; a middle box keeps that shape; a readout scores 4 letters at each position, `[3×4]`. Arrows carry
shapes; boxes hold counts: 8, 6 and 4. More letters change the shapes, never the counts. (The page's machine is made up to show
the idea.)

### M2.1.2 · Watch it on our line

*(The page's recorded clip, in words.)*

Open the box. Between the 32 ids that go in and the 65 probabilities that come out sit five stages: a token table and a
position table; one block; a final LayerNorm, which rescales each row; and a readout, which scores every character. Every arrow
between them carries a tensor with a shape: `[32]`, then `[32×28]` through the block, then `[32×65]` at the readout. The
learned numbers sit in the parts: 2,716 in the two tables, 6,580 in the block, 56 in the final LayerNorm and
65 biases in the readout, 9,417 in all. This plan, with every shape and count, is the **blueprint**.

### M2.1.3 · Read and run the code

Counting the learned numbers is `learned_numbers` in `code/inside_embeddings.py` (copied in above): `count` adds up every
weight and bias in one part, then the parts are listed in the order the forward pass meets them. The cell walks our line
through the five stages and prints each shape, then counts each part: 1,820, 896, 6,580, 56 and 65, 9,417 in
all.

In [6]:
x = embed(model, ids);                                  print('the two tables  ', tuple(x.shape))        # (32, 28)
x = model.blocks(x[None])[0];                           print('one block       ', tuple(x.shape))        # (32, 28)
x = model.ln_f(x);                                      print('final LayerNorm ', tuple(x.shape))        # (32, 28)
logits = x @ model.tok_emb.weight.T + model.head_bias;  print('readout         ', tuple(logits.shape))   # (32, 65)
p = torch.softmax(logits[-1], dim=-1);                  print('probabilities   ', tuple(p.shape))        # (65,): the last row's
assert torch.allclose(logits, model(ids[None])[0][0], atol=1e-5)    # stage by stage, the same as the model's own forward pass
assert tuple(ids.shape) == (32,) and tuple(x.shape) == (32, 28) and tuple(logits.shape) == (32, 65) and tuple(p.shape) == (65,)

parts = learned_numbers(model)
for name, n in parts.items():
    print(f"  {name:16s} {n:5,}")
print(f"  {'all':16s} {sum(parts.values()):5,}")
assert parts['token table'] + parts['position table'] == 2716 and parts['block'] == 6580
assert parts['final LayerNorm'] == 56 and parts['readout bias'] == 65 and sum(parts.values()) == count_params(model) == 9417

the two tables   (32, 28)
one block        (32, 28)
final LayerNorm  (32, 28)
readout          (32, 65)
probabilities    (65,)
  token table      1,820
  position table     896
  block            6,580
  final LayerNorm     56
  readout bias        65
  all              9,417


### ✋ Your turn

Count the block's learned numbers yourself: add up `p.numel()` for every `name, p` in `model.named_parameters()` whose name starts with `'blocks.'`, and put the total in `answer`.

In [7]:
answer = None   # TODO: sum(p.numel() for name, p in model.named_parameters() if name.startswith(...))

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == learned_numbers(model)["block"], "add p.numel() over the parameters whose name starts with 'blocks.'"
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [8]:
#@title Show a solution { display-mode: "form" }
answer = sum(p.numel() for name, p in model.named_parameters() if name.startswith('blocks.'))
print('the block holds', answer, 'of', count_params(model), 'learned numbers')

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == learned_numbers(model)["block"], "add p.numel() over the parameters whose name starts with 'blocks.'"
    print("✓ right:", answer)

the block holds 6580 of 9417 learned numbers
✓ right: 6580


## M2.2 · The token table

*How does an id become a vector?*

**Predict first** (the page asks it too): our line has 32 characters, but only 16 different ones. How many different
entries of the token table does it fetch: 32, 16, or 65?

### M2.2.1 · Picture it

To compute, each character needs numbers of its own, kept in a table. A made-up alphabet of 5 letters, ids 0 to 4, and a table
with one entry of 3 numbers for each. The word *bad* is ids 1, 0 and 3: each id fetches its entry, in order. Nothing is
calculated: the id only says which entry to copy, like a locker number. 3 ids in, 3 rows of 3 out.

### M2.2.2 · Watch it on our line

*(The page's recorded clip, in words.)*

The **token embedding table** holds one entry for each of the 65 characters: a **vector**, an ordered list of 28 learned
numbers. So the table is `[65×28]`, 1,820 numbers. Each id fetches its entry: capital Y, id 37, fetches entry 37.
Our 32 ids become 32 vectors, `[32]` → `[32×28]`. From here on, the vector each position holds is its **row**.

### M2.2.3 · Read and run the code

The lookup is `token_rows` in `code/inside_embeddings.py`: it takes the token table, `[65×28]`, and indexes it with the ids,
so row `t` is entry `ids[t]`. The cell runs it on our line, checks that row 0 is entry 37, capital Y's, counts the
different entries it fetches, and compares its
rows with the recorded run's.

In [9]:
table = model.tok_emb.weight
print('token table', tuple(table.shape), '=', table.numel(), 'learned numbers')      # (65, 28) = 1820
rows = token_rows(model, ids)
print('ids', tuple(ids.shape), '→ rows', tuple(rows.shape))                         # (32,) → (32, 28)
print("Y is id", tok.stoi['Y'], "; row 0 is entry", tok.stoi['Y'], ":", torch.equal(rows[0], table[tok.stoi['Y']]))
print('row 0 starts', [round(v, 4) for v in rows[0, :4].tolist()], '…')
print('different entries fetched:', len(set(ids.tolist())))                    # 16
assert len(set(ids.tolist())) == 16
assert tuple(table.shape) == (65, 28) and table.numel() == 1820 and tuple(rows.shape) == (32, 28)
assert tok.stoi['Y'] == 37 and torch.equal(rows[0], table[37])
# the recorded run's token rows (the course's export): rows 0 and 31, their first 4 numbers
assert torch.allclose(rows[0, :4], torch.tensor([0.8497, -0.5515, -0.6307, -0.7595]), atol=2e-4)
assert torch.allclose(rows[31, :4], torch.tensor([-0.4318, 0.3458, 0.4031, 0.0633]), atol=2e-4)

token table (65, 28) = 1820 learned numbers
ids (32,) → rows (32, 28)
Y is id 37 ; row 0 is entry 37 : True
row 0 starts [0.8497, -0.5515, -0.6307, -0.7595] …
different entries fetched: 16


The four a’s in our line all fetch entry 39: four identical vectors.

In [10]:
a_at = [i for i, c in enumerate(line) if c == 'a']
print("the a's sit at positions", a_at, "and all have id", tok.stoi['a'])
for i in a_at:
    print(f"  row {i:2d}:", [round(v, 3) for v in rows[i, :6].tolist()], '…')
assert a_at == [5, 15, 19, 24] and tok.stoi['a'] == 39
assert all(torch.equal(rows[i], rows[a_at[0]]) for i in a_at)        # four identical rows

the a's sit at positions [5, 15, 19, 24] and all have id 39
  row  5: [-0.139, 0.104, -0.336, -0.625, -0.12, 0.123] …
  row 15: [-0.139, 0.104, -0.336, -0.625, -0.12, 0.123] …
  row 19: [-0.139, 0.104, -0.336, -0.625, -0.12, 0.123] …
  row 24: [-0.139, 0.104, -0.336, -0.625, -0.12, 0.123] …


### ✋ Your turn

The line has more spaces than a’s. Put the positions of all its spaces in `answer`, as a list. Do their token rows match, too?

In [11]:
answer = None   # TODO: [i for i, c in enumerate(line) if ...]

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == [i for i, c in enumerate(line) if c == ' '], "the positions where line[i] is a space, ' '"
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [12]:
#@title Show a solution { display-mode: "form" }
answer = [i for i, c in enumerate(line) if c == ' ']
print('all the same row:', all(torch.equal(rows[i], rows[answer[0]]) for i in answer))

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == [i for i, c in enumerate(line) if c == ' '], "the positions where line[i] is a space, ' '"
    print("✓ right:", answer)

all the same row: True
✓ right: [3, 10, 13, 23, 27, 30]


## M2.3 · What the entries learned

*What did the entries learn?*

**Predict first** (the page asks it too): after training, which entry of the token table ends up nearest the semicolon's: the
space's, since it comes right after a semicolon; the comma's, since they turn up in the same kinds of places; or small i's,
since it looks most alike?

### M2.3.1 · Picture it

Watch a made-up table learn: which entries end up alike? Imagine entries of 2 numbers, so each is an arrow on a page. In a
made-up text, *x* and *y* always come before *o*, and *z* before *q*. Each training step nudges the arrows of *x* and *y*
the same way, since both must predict *o*; the arrow of *z* drifts apart. Closeness is measured by direction: arrows pointing
the same way have a similarity of 1, at right angles 0, opposite −1.

### M2.3.2 · Watch it on our line

*(The page's recorded clip, in words.)*

Nobody designed the token table: its 1,820 numbers started random, and training set them. To see what training put there,
measure how alike two entries are with **cosine similarity**: 1 means they point the same way, 0 means unrelated. The
semicolon's nearest entry is the comma, at 0.97; the full stop's is the question mark, at 0.99. The
space sits nearest the new line, at 0.68; small e nearest small o, at 0.41. Characters used in
similar places ended up with similar entries: learned from the text, not written in.

### M2.3.3 · Read and run the code

The nearest entries come from `nearest` in `code/inside_embeddings.py`: it scales every entry to length 1, one multiplication
gives every entry's similarity to the chosen one (its cosine), the character itself is left out, and the 3 largest are kept. The cell runs
it for five characters and checks each against the recorded run.

In [13]:
for ch in [';', '.', ' ', 'e', 'A']:
    print(f"{ch!r:5} nearest:", nearest(model, tok, ch))
# the recorded run's nearest entries (the course's export)
assert nearest(model, tok, ';')[0][0] == ',' and abs(nearest(model, tok, ';')[0][1] - 0.97) < 2e-4
assert nearest(model, tok, '.')[0][0] == '?' and abs(nearest(model, tok, '.')[0][1] - 0.9893) < 2e-4
assert nearest(model, tok, ' ')[0][0] == '\n' and abs(nearest(model, tok, ' ')[0][1] - 0.6814) < 2e-4
assert nearest(model, tok, 'e')[0][0] == 'o' and abs(nearest(model, tok, 'e')[0][1] - 0.4118) < 2e-4
assert nearest(model, tok, 'A')[0][0] == 'B' and abs(nearest(model, tok, 'A')[0][1] - 0.5628) < 2e-4

';'   nearest: [(',', 0.97), ('.', 0.9622), ('?', 0.9534)]
'.'   nearest: [('?', 0.9893), ('!', 0.9723), (';', 0.9622)]
' '   nearest: [('\n', 0.6814), ('-', 0.6411), (',', 0.3977)]
'e'   nearest: [('o', 0.4118), ('a', 0.4029), ('i', 0.3102)]
'A'   nearest: [('B', 0.5628), ('O', 0.5501), ('E', 0.4947)]


Before training, the same table held small random numbers. The recorded run started from seed 1337, so the cell below rebuilds
that starting table exactly, and counts the pairs of entries with a similarity above 0.9, before and after training.

In [14]:
torch.manual_seed(1337)                            # the recorded run started from these random numbers
start = Prompter(cfg)
assert torch.allclose(start.tok_emb.weight[0, :4], torch.tensor([0.0049, -0.0100, -0.0179, 0.0106]), atol=1e-4)   # the recorded starting table

def alike(m, above=0.9):
    unit = F.normalize(m.tok_emb.weight, dim=1)                       # [65 × 28], every entry at length 1
    cos = unit @ unit.T                                               # [65 × 65]: every pair's cosine similarity
    i, j = torch.triu_indices(len(cos), len(cos), 1)                  # each pair once
    return [(tok.chars[a], tok.chars[b]) for a, b in zip(i.tolist(), j.tolist()) if cos[a, b] > above]

before, after = alike(start), alike(model)
print('pairs above 0.9 before training:', len(before))
print('pairs above 0.9 after training: ', len(after), after)
assert len(before) == 0 and len(after) == 9 and set(''.join(a + b for a, b in after)) == set(',;.?!')

pairs above 0.9 before training: 0
pairs above 0.9 after training:  9 [('!', '.'), ('!', ';'), ('!', '?'), (',', '.'), (',', ';'), (',', '?'), ('.', ';'), ('.', '?'), (';', '?')]


A flat map shows the same thing at a glance: squash each entry of 28 numbers onto the two directions along which the 65
entries spread most. The map keeps only about 25 percent of the spread, so closeness on it is rough; the cosine above is
the exact measure. Capitals gather on one side, small letters on another, the marks in a corner.

In [15]:
E = model.tok_emb.weight.double()
Z = E - E.mean(0)                                         # [65 × 28], centred
_, S, Vt = torch.linalg.svd(Z, full_matrices=False)       # the directions of largest spread, largest first
xy = Z @ Vt[:2].T                                         # [65 × 2]: each entry's place on the flat map
xy = xy * torch.sign(xy[tok.stoi['a']])                   # turned so that small a sits up and to the right, as on the page
assert torch.allclose(xy[tok.stoi['a']].float(), torch.tensor([0.8270, 0.6883]), atol=2e-4)   # the recorded map
assert torch.allclose(xy[tok.stoi['Y']].float(), torch.tensor([-0.8433, -0.2349]), atol=2e-4)
kept = ((S ** 2)[:2].sum() / (S ** 2).sum()).item()
print(f"the flat map keeps {100 * kept:.0f}% of the spread")
assert round(100 * kept) == 25

W, H = 64, 20                                             # a text picture: where two characters land on one spot, one shows
lo, hi = xy.min(0).values, xy.max(0).values
picture = [[' '] * W for _ in range(H)]
for i, c in enumerate(tok.chars):
    col = int(round(((xy[i, 0] - lo[0]) / (hi[0] - lo[0]) * (W - 1)).item()))
    row = int(round(((hi[1] - xy[i, 1]) / (hi[1] - lo[1]) * (H - 1)).item()))      # up the page is up the map
    picture[row][col] = {'\n': '↵', ' ': '␣'}.get(c, c)
print('\n'.join(''.join(r) for r in picture))

the flat map keeps 25% of the spread
                                                 b              
                                                 p              
                                       v         f       r      
             B                       j    m       hc ae         
                                       z     x  y w  doi  t     
   R                  T                                  n      
     A      HF                           q      gu    k         
    P    M  G I                                         s       
        N     S       W   $                                     
X    E D    O  Y                                           ␣    
 Q    Z   U      J                                        -     
                                  &                             
            K                                                   
                                                     ↵          
                                                     

### ✋ Your turn

Which entry is nearest to small `a`? Put that character in `answer`.

In [16]:
answer = None   # TODO: nearest(model, tok, 'a') lists the closest three

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == nearest(model, tok, 'a')[0][0], "the first character that nearest(model, tok, 'a') lists"
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [17]:
#@title Show a solution { display-mode: "form" }
answer = nearest(model, tok, 'a')[0][0]
print(nearest(model, tok, 'a'))

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == nearest(model, tok, 'a')[0][0], "the first character that nearest(model, tok, 'a') lists"
    print("✓ right:", answer)

[('e', 0.4029), ('o', 0.3221), ('A', 0.2759)]
✓ right: e


# Section 2 · Where each character sits

Each row gets its place in the line, and the rows leave the first stage: the position table, then the residual stream.

## M2.4 · The position table

*How does each row get its place in the line?*

**Predict first** (the page asks it too, and its home page holds it from the start): our line has four a’s, at positions
5, 15, 19, 24. Do they start out as the same numbers inside the model, or four different ones? The same, and they
stay the same; the same at first, then four rows with nothing left in common; or the same at first, then four different rows that
stay alike?

### M2.4.1 · Picture it

Swap two letters and the word changes: n, o spells *no*, but o, n spells *on*. In a made-up table, the word *no* fetches the
entry of n, then of o; the word *on* fetches the entry of o, then of n. The same two rows, only swapped: neither row says which seat it
holds. So give each seat, each position, an entry of its own: seat 0 and seat 1, learned like the letters'.

### M2.4.2 · Watch it on our line

*(The page's recorded clip, in words.)*

Each row now says which character sits there, but nothing yet says where. A second table, the **position embedding**, holds one
learned entry per position, 0 to 31: `[32×28]`, 896 numbers, and each row gets its position's entry added, number by
number. Now the four a’s, at positions 5, 15, 19 and 24, have four different rows: their
similarity drops from exactly 1 to between 0.85 and 0.99. So the answer to the Predict: the same numbers at first, four
different rows once the positions are added. Each row now holds what the character is, plus where it sits.

### M2.4.3 · Read and run the code

The position table is read by `position_rows` in `code/inside_embeddings.py`, which keeps one entry for each position of the
line, and added by `embed`, which fetches the token rows and adds the position rows, number by number. The cell runs it on our
line and compares the four a's: similarities from 0.85 to 0.99, where the token rows alone gave exactly 1.

In [18]:
pos = position_rows(model, len(ids))
print('position table', tuple(model.pos_emb.weight.shape), '=', model.pos_emb.weight.numel(), 'learned numbers')   # (32, 28) = 896
x0 = embed(model, ids)
print('token rows', tuple(rows.shape), '+ position rows', tuple(pos.shape), '=', tuple(x0.shape))
assert model.pos_emb.weight.numel() == 896 and tuple(x0.shape) == (32, 28) and torch.equal(x0, rows + pos)
# the recorded run's position rows and their sums (the course's export): row 5, the first a, and row 31
assert torch.allclose(pos[5, :4], torch.tensor([-0.2687, 0.0205, 0.0560, -0.0214]), atol=2e-4)
assert torch.allclose(x0[5, :4], torch.tensor([-0.4075, 0.1248, -0.2795, -0.6465]), atol=2e-4)
assert torch.allclose(x0[31, :4], torch.tensor([-0.5172, 0.5992, 0.6395, 0.2567]), atol=2e-4)

for i in a_at:
    print(f"  a at {i:2d}:", [round(v, 3) for v in x0[i, :6].tolist()], '…')
sims = {(i, j): F.cosine_similarity(x0[i], x0[j], dim=0).item() for k, i in enumerate(a_at) for j in a_at[k + 1:]}
for (i, j), s in sims.items():
    print(f"  similarity of the a's at {i:2d} and {j:2d}: {s:.3f}")
lo, hi = min(sims.values()), max(sims.values())
print(f"before the positions: exactly 1.  after: between {lo:.2f} and {hi:.2f}")
assert round(lo, 2) == 0.85 and round(hi, 2) == 0.99 and hi < 0.999

position table (32, 28) = 896 learned numbers
token rows (32, 28) + position rows (32, 28) = (32, 28)
  a at  5: [-0.408, 0.125, -0.28, -0.647, -0.263, -0.016] …
  a at 15: [-0.372, 0.202, -0.187, -0.585, -0.153, 0.027] …
  a at 19: [-0.34, 0.244, -0.153, -0.537, -0.108, 0.071] …
  a at 24: [-0.278, 0.31, -0.106, -0.473, -0.068, 0.12] …
  similarity of the a's at  5 and 15: 0.959
  similarity of the a's at  5 and 19: 0.920
  similarity of the a's at  5 and 24: 0.852
  similarity of the a's at 15 and 19: 0.993
  similarity of the a's at 15 and 24: 0.962
  similarity of the a's at 19 and 24: 0.988
before the positions: exactly 1.  after: between 0.85 and 0.99


### ✋ Your turn

Which two a’s have the most similar rows now? Put their positions in `answer`, as a pair like `(5, 15)`. Are they near each other in the line?

In [19]:
answer = None   # TODO: the pair (i, j) with the largest value in sims

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == max(sims, key=sims.get), 'the key of sims with the largest similarity: max(sims, key=sims.get)'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [20]:
#@title Show a solution { display-mode: "form" }
answer = max(sims, key=sims.get)
print('positions', answer, 'are', answer[1] - answer[0], 'apart; the least alike pair is', min(sims, key=sims.get))

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == max(sims, key=sims.get), 'the key of sims with the largest similarity: max(sims, key=sims.get)'
    print("✓ right:", answer)

positions (15, 19) are 4 apart; the least alike pair is (5, 24)
✓ right: (15, 19)


## M2.5 · The rows that leave

*What leaves this stage?*

**Predict first** (the page asks it too): our line leaves this stage as 32 rows. Then another line comes in. What changes: only
the rows, the two tables staying as training left them; the tables too, adjusting to each new line; or nothing, each
character’s row stored once, ready to fetch?

### M2.5.1 · Picture it

Many stages come after this one. How does each one's work reach the next? A made-up line of 3 characters, as 3 rows riding a
belt through the stages. Each stage reads the rows, works something out, and adds its result onto them, number by number. So no
row grows or is replaced, and the last stage reads all that was added.

### M2.5.2 · Watch it on our line

*(The page's recorded clip, in words.)*

The first stage hands on 32 rows of 28 numbers, `[32×28]`: the **residual stream**, the running rows that every later stage
reads, and that the block adds to. Nothing else is handed on: given only these rows, the rest of the model computes exactly the probabilities
the whole model gives (its top choice after our line is `r`, at 0.340). Learned numbers so far: 1,820 plus
896, 2,716 of 9,417. The numbers made fresh for this input are the 32 rows: another line gives other rows, from
the same two tables.

### M2.5.3 · Read and run the code

The whole pass is `forward` in `training/prompter/model.py` (copied in above): its next line makes the rows, the token rows
plus the position rows; the rows go through the block and the final LayerNorm, keeping their shape, `[32×28]`; only then does
the readout score every character. The cell hands our rows to everything after this stage and gets its guess after our line:
`r` first, at 0.340.

In [21]:
print('the residual stream', tuple(x0.shape))
rest = model.ln_f(model.blocks(x0[None])) @ model.tok_emb.weight.T + model.head_bias     # every stage after this one
assert torch.allclose(rest, model(ids[None])[0], atol=1e-5)                              # x0 is all the later stages need
p = torch.softmax(rest[0, -1], dim=-1)
top = p.topk(5)
print('top 5 after the line:', [(tok.chars[i], round(v, 4)) for v, i in zip(top.values.tolist(), top.indices.tolist())])
assert [tok.chars[i] for i in top.indices.tolist()] == ['r', 'a', 'l', 'o', 'e']
assert torch.allclose(top.values, torch.tensor([0.3403, 0.2101, 0.1328, 0.1061, 0.1044]), atol=2e-4)

learned = parts['token table'] + parts['position table']
print(f"learned so far: {parts['token table']:,} + {parts['position table']:,} = {learned:,} of {count_params(model):,}")
assert learned == 2716
other = embed(model, tok.encode("O Romeo, Romeo! wherefore art th"))
print('another line:', tuple(other.shape), '; the same rows?', torch.equal(other, x0))

the residual stream (32, 28)
top 5 after the line: [('r', 0.3403), ('a', 0.2101), ('l', 0.1328), ('o', 0.1061), ('e', 0.1044)]
learned so far: 1,820 + 896 = 2,716 of 9,417
another line: (32, 28) ; the same rows? False


### ✋ Your turn

Change the line: embed only its last 8 characters, `'and my p'`. What shape comes out? Put `tuple(...shape)` in `answer`. Then compare: is this p's row the same as row 31 of the whole line?

In [22]:
answer = None   # TODO: tuple(embed(model, tok.encode('and my p')).shape)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == (8, 28), "embed(model, tok.encode('and my p')) has one row of 28 per character"
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [23]:
#@title Show a solution { display-mode: "form" }
short = embed(model, tok.encode('and my p'))
answer = tuple(short.shape)
print('the p of the short line sits at position', len(short) - 1, '; the same row as position 31?', torch.equal(short[-1], x0[31]))

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == (8, 28), "embed(model, tok.encode('and my p')) has one row of 28 per character"
    print("✓ right:", answer)

the p of the short line sits at position 7 ; the same row as position 31? False
✓ right: (8, 28)
